## install dependencies:

In [30]:
!pip install -q langchain-groq

##Environment setup cell

In [31]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()



os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
print("✅ LLM ready!")

✅ LLM ready!


##Classifier agent (the router)

In [32]:
classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a topic classifier. Given a topic, reply with ONLY one word:
- 'tech' if the topic is about technology, AI, programming, software, hardware, science, or engineering
- 'general' if it's about anything else (business, lifestyle, health, education, culture, etc.)

Reply with ONLY 'tech' or 'general'. No punctuation, no explanation."""),
    ("human", "Topic: {topic}")
])

classifier_chain = classifier_prompt | llm | StrOutputParser()

def classify_topic(topic: str) -> str:
    result = classifier_chain.invoke({"topic": topic})
    return result.strip().lower()

# Test
print(classify_topic("AI in Healthcare"))     # → tech
print(classify_topic("Remote Work Productivity"))  # → general

tech
general


## Tech Writer agent

In [33]:
tech_writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert Tech Writer who creates professional LinkedIn posts.
Your posts are insightful, data-aware, and speak to professionals in the tech industry.

Rules:
- Write 2–4 short paragraphs
- Use a professional, engaging tone
- Reference relevant industry trends or facts when possible
- End with a thought-provoking question or call-to-action
- Write entirely in the specified language"""),
    ("human", "Write a LinkedIn post about: {topic}\nLanguage: {language}")
])

tech_writer_chain = tech_writer_prompt | llm | StrOutputParser()

## General Writer agent

In [34]:
general_writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional LinkedIn content creator who specialises in non-technical topics.
Your writing is warm, relatable, and inspires reflection or action.

Rules:
- Write 2–4 short paragraphs
- Use a professional but approachable tone
- Include a real insight or perspective, not generic advice
- End with a thoughtful question or call-to-action
- Write entirely in the specified language"""),
    ("human", "Write a LinkedIn post about: {topic}\nLanguage: {language}")
])

general_writer_chain = general_writer_prompt | llm | StrOutputParser()

## Routing logic (conditional handover)

In [35]:
def generate_linkedin_post(topic: str, language: str) -> dict:
    """Main agent function with conditional routing."""

    print(f"\n📋 Topic: {topic}")
    print(f"🌐 Language: {language}")

    # Step 1: Classify
    category = classify_topic(topic)
    print(f"🔀 Classified as: {category.upper()}")

    # Step 2: Route to the correct writer
    if category == "tech":
        print("⚙️  Routing to Tech Writer Agent...")
        post = tech_writer_chain.invoke({"topic": topic, "language": language})
    else:
        print("💼 Routing to General Writer Agent...")
        post = general_writer_chain.invoke({"topic": topic, "language": language})

    return {
        "topic": topic,
        "language": language,
        "category": category,
        "post": post
    }

## Demo with two required examples

In [36]:
# ── Example 1: Tech topic in English ──────────────────────────────
result1 = generate_linkedin_post(
    topic="AI in Healthcare",
    language="English"
)
print("\n" + "="*60)
print("GENERATED POST:")
print("="*60)
print(result1["post"])

# ── Example 2: General topic in Bengali ───────────────────────────
result2 = generate_linkedin_post(
    topic="Remote Work Productivity",
    language="Bengali"
)
print("\n" + "="*60)
print("GENERATED POST:")
print("="*60)
print(result2["post"])


📋 Topic: AI in Healthcare
🌐 Language: English
🔀 Classified as: TECH
⚙️  Routing to Tech Writer Agent...

GENERATED POST:
As we continue to navigate the complexities of the healthcare industry, it's becoming increasingly clear that Artificial Intelligence (AI) is poised to play a transformative role in shaping the future of patient care. From diagnosing diseases more accurately to streamlining clinical workflows, AI is revolutionizing the way healthcare professionals approach their work. According to a recent report, the global AI in healthcare market is expected to reach $35.9 billion by 2027, growing at a CAGR of 41.8% during the forecast period.

One of the most exciting applications of AI in healthcare is in the realm of medical imaging. AI-powered algorithms can analyze vast amounts of medical image data, helping radiologists and clinicians to identify abnormalities and diagnose diseases more quickly and accurately. For instance, AI-assisted computer vision can help detect breast 

## README

# AI-Powered LinkedIn Post Generator

## Agent Workflow
1. **User Input** → Topic string + Language choice
2. **Classifier Agent** → Reads the topic, returns 'tech' or 'general'
3. **Conditional Router** → Python if/else sends to the right writer
4. **Tech Writer Agent** → Generates post for technology topics
5. **General Writer Agent** → Generates post for non-tech topics
6. **Output** → 2–4 paragraph LinkedIn post in the selected language

## Routing Logic
The classifier uses a zero-shot LLM call with a strict system prompt.
It returns exactly one token ('tech' or 'general'), which is then used
in a simple if/else to hand off to the appropriate writer chain.

## Example Inputs
- "AI in Healthcare" + English → Tech Writer
- "Remote Work Productivity" + Bengali → General Writer